In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_tuner as kt
import keras
from keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    r2_score, mean_squared_error, root_mean_squared_error,
    mean_absolute_error, mean_absolute_percentage_error, RocCurveDisplay
)
from sklearn.inspection import PartialDependenceDisplay

In [ ]:
# Creating a function to prep the raw data
def prep_rocket_league_data(file):
    # Read in the data
    # Drop assists since it is 0 for 1v1; drop mvp since there is always 1 mvp per match
    matches = pd.read_csv(f'./{file}').drop(columns=['assists','mvp'])

    # First, we'll pivot the data so that we preserve the individual player stats for each match.
    matches_subset = matches.drop(columns=['map_code','rank','duration'], errors='ignore')
    matches_subset_pivoted = matches_subset.pivot(index='match_id', columns='color', values=[col for col in matches_subset.columns if col != 'match_id' and col != 'color'])
    matches_subset_pivoted.columns = matches_subset_pivoted.columns.map(lambda x: f"{x[0]}_{x[1]}")

    # Make sure numeric columns have a numeric data type
    cols_to_convert = [col for col in matches_subset_pivoted.columns if col not in ['match_id','car_name_blue','car_name_orange']]
    matches_subset_pivoted[cols_to_convert] = matches_subset_pivoted[cols_to_convert].astype('float64')

    # Next we'll aggregate the original matches data so that we have 1 row per match containing both the mean and difference of all numeric variables.
    # For map_code, we'll just use max since it is the same value for both players.
    # For all numeric variables, we'll take both the mean and difference.

    # Define aggregation
    agg_dict = {col: ["mean",(lambda x: x.max() - x.min())] for col in matches.select_dtypes('number').columns if col != 'match_id'}
    agg_dict.update({"map_code": "max"})

    # Apply aggregation
    if 'rank' in matches.columns:
        matches_prepped = matches.drop(columns=['car_name']).groupby(['match_id','rank']).agg(agg_dict).reset_index()
    else:
        matches_prepped = matches.drop(columns=['car_name']).groupby('match_id').agg(agg_dict).reset_index()

    # Rename columns
    matches_prepped.columns = matches_prepped.columns.map(lambda x: f"{x[0]}_{x[1]}".replace("_<lambda_0>", "_diff").strip("_"))

    # Dropping duration_diff since it is 0 across the board and rename duration_mean to just duration.
    # Also rename map_code_max to map_code for clarity
    matches_prepped = matches_prepped.drop(columns=['duration_diff']).rename(columns={'map_code_max':'map_code','duration_mean':'duration'})

    # Finally we'll merge the 2 prepared dataframes so that we have individual player stats and aggregated match stats for each match.
    matches_prepped_final = pd.merge(matches_subset_pivoted, matches_prepped, how='inner', on='match_id')

    return matches_prepped_final

In [ ]:
# Prep the labeled data
matches_prepped_final = prep_rocket_league_data('train.csv')

In [ ]:
# Our features are everything except for match id and rank
variables = [col for col in matches_prepped_final.columns if col != 'match_id' and col != 'rank']

X = matches_prepped_final[variables]
y = matches_prepped_final['rank']

le = LabelEncoder()
y = le.fit_transform(y)

# Split data into test and train so we can evaluate how the model does
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 0, stratify=y)

In [ ]:
# We'll apply a SimpleImputer and a StandardScaler to numeric features
numeric_features = [col for col in matches_prepped_final.select_dtypes('number').columns if col != 'match_id']
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(missing_values=np.nan, strategy='mean')),
        ("scaler", StandardScaler())]
)

# We'll apply a one-hot encoder to the categorical features
categorical_features = [col for col in matches_prepped_final.select_dtypes('object').columns if col != 'rank']
categorical_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Append classifier to preprocessing pipeline. Now we have a full prediction pipeline.
nn_pipe = Pipeline(
    steps=[("preprocessor", preprocessor)]
)

# Fit and transform the training data, and then transform the test data in the same way
X_train = nn_pipe.fit_transform(X_train)
X_test = nn_pipe.transform(X_test)

In [ ]:
# Define the model building function.
# We'll try different parameters for learning rate, units (number of neurons in the hidden layer), and activation function
def build_model(hp):
    model = tf.keras.Sequential()

    # Input layer
    model.add(keras.layers.InputLayer(input_shape = (X_test.shape[1],)))

    # Hidden layer
    model.add(keras.layers.Dense(units=hp.Int('units_1', min_value=8, max_value=128, step=8),
                                 activation=hp.Choice('dense_activation_1', values=['tanh','relu'])))

    # Output layer
    model.add(tf.keras.layers.Dense(6, activation = 'softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])

    return model

In [ ]:
# Instantiate the tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=25,
    executions_per_trial=1,
    directory='my_dir',
    project_name='keras_tuner_demo',
    overwrite=True)

# Run the search
tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

Trial 25 Complete [00h 00m 13s]
val_loss: 1.0085123777389526

Best val_loss So Far: 1.0016305446624756
Total elapsed time: 00h 06m 21s


In [ ]:
# Retrieve the best hyperparameters
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best units (layer 1): {best_hp.get('units_1')}")
print(f"Best activation function (layer 1): {best_hp.get('dense_activation_1')}")
print(f"Best learning rate: {best_hp.get('learning_rate')}")


Best units (layer 1): 56
Best activation function (layer 1): tanh
Best learning rate: 0.0015630688472243843


In [ ]:
# Build and train the optimized model with the optimal hyperparameters
model = tuner.hypermodel.build(best_hp)
es = EarlyStopping(monitor='val_loss', patience=3)
model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks = [es])

Epoch 1/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4733 - loss: 1.1839 - val_accuracy: 0.5196 - val_loss: 1.0671
Epoch 2/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5585 - loss: 0.9937 - val_accuracy: 0.5424 - val_loss: 1.0336
Epoch 3/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5820 - loss: 0.9447 - val_accuracy: 0.5385 - val_loss: 1.0318
Epoch 4/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6051 - loss: 0.9038 - val_accuracy: 0.5579 - val_loss: 1.0184
Epoch 5/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6218 - loss: 0.8736 - val_accuracy: 0.5415 - val_loss: 1.0341
Epoch 6/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6377 - loss: 0.8463 - val_accuracy: 0.5433 - val_loss: 1.0238
Epoch 7/100
291/291 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6554 - loss: 0.8183 - val_accuracy: 0.5489 - val_loss: 1.0317


In [ ]:
# Calculating predicted ranks

# Generate predictions on the test set by applying the model
y_pred_raw = model.predict(X_test)

# Get the most probable rank for each match
y_pred = np.argmax(y_pred_raw, axis = 1)

121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


We'll look at how the model did using the confusion matrix and classification report.

In [ ]:
print(confusion_matrix(y_test, y_pred))

[[ 18   0   2  12   3  35]
 [  1 528 191   2  30   0]
 [  2 216 413  32 226   3]
 [ 10   1  28 459 206 108]
 [  1  38 209 220 499  14]
 [ 14   0   2 150  24 174]]


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.39      0.26      0.31        70
           1       0.67      0.70      0.69       752
           2       0.49      0.46      0.48       892
           3       0.52      0.57      0.54       812
           4       0.51      0.51      0.51       981
           5       0.52      0.48      0.50       364

    accuracy                           0.54      3871
   macro avg       0.52      0.50      0.50      3871
weighted avg       0.54      0.54      0.54      3871



In [ ]:
# Now fit the model on all available training data (using the optimal hyperparameters)
# First perform the preprocessing steps
X = nn_pipe.fit_transform(X)

# Redefine the model based on the number of features in X
model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape = (X.shape[1],)))

model.add(keras.layers.Dense(units=56, activation='tanh'))

model.add(tf.keras.layers.Dense(6, activation = 'softmax'))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0015630688472243843),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

# Then fit the model
history = model.fit(X, y, epochs = 500, validation_split = 0.2, callbacks = [es])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/500
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5005 - loss: 1.1390 - val_accuracy: 0.5082 - val_loss: 1.0637
Epoch 2/500
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5643 - loss: 0.9817 - val_accuracy: 0.5170 - val_loss: 1.0385
Epoch 3/500
388/388 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5886 - loss: 0.9372 - val_accuracy: 0.5241 - val_loss: 1.0435


In [ ]:
# Prep unlabeled data
test_prepped_final = prep_rocket_league_data('test.csv')

# Define variables
variables = [col for col in test_prepped_final.columns if col != 'match_id']

# Generate predictions on the test set by applying the model
X_test_processed = nn_pipe.transform(test_prepped_final[variables])
y_pred_raw = model.predict(X_test_processed)

# Get the most probable rank for each match
y_pred_labels = np.argmax(y_pred_raw, axis = 1)
# Reverse transform the rank so that it is a string
y_pred = le.inverse_transform(y_pred_labels)

# Convert string ranks to numeric
converter = { 'bronze': 1, 'silver': 2, 'gold': 3, 'platinum': 4, 'diamond': 5, 'champion': 6 }
y_pred = pd.Series(y_pred).map(converter)

# Concatenate match ids with rank predictions to get final submission
submission = pd.concat([test_prepped_final['match_id'].astype('int'), y_pred], axis = 1).rename(columns = {0: 'rank'})

# Save to csv
submission.to_csv('./submission4.csv', index = False)

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
